In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# RoundRobin 으로 실행될 때 종료 지점을 알려줘야 하는데 이때 MaxMessage, TextMention 을 사용한다.
# 채팅이 맥스 메시지 수를 초과하거나 특정 텍스트를 포함하는 메시지가 있을 때 종료된다.
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from dotenv import load_dotenv

load_dotenv()

In [ ]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""당신은 명확성과 간결함에 집중하는 전문 편집자입니다.
            모든 응답은 반드시 한국어로 작성하세요.
            모호함과 중복을 제거하고, 모든 문장을 간결하고 명확하게 만드는 것이 당신의 역할입니다.
            설득력이나 어조는 신경 쓰지 말고, 메시지를 읽기 쉽고 이해하기 쉽게 만드세요.""",
)

tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""당신은 감정적 어조와 전문성에 집중하는 커뮤니케이션 코치입니다.
            모든 응답은 반드시 한국어로 작성하세요.
            이메일이 따뜻하고, 자신감 있으며, 인간적인 느낌을 주도록 — 동시에 전문적이고
            대상에 적합하게 만드는 것이 당신의 역할입니다. 감정적 공감을 높이고, 표현을 다듬으며,
            딱딱하거나 차갑거나 지나치게 캐주얼하게 들리는 단어를 조정하세요.""",
)

persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""당신은 마케팅, 행동 심리학, 카피라이팅에 정통한 설득 전문가입니다.
            모든 응답은 반드시 한국어로 작성하세요.
            이메일의 설득력을 높이는 것이 당신의 역할입니다: 행동 유도 문구를 개선하고, 논거를 구조화하며, 이점을 강조하세요. 약하거나 수동적인 표현은 제거하세요.""",
)

synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""당신은 고급 이메일 작성 전문가입니다.
            모든 응답은 반드시 한국어로 작성하세요.
            이전 에이전트들의 응답과 수정안을 모두 읽고, **최선의 아이디어를 종합**하여
            통일된 완성도 높은 이메일 초안을 만드는 것이 당신의 역할입니다. 다음에 집중하세요:
            명확성, 어조, 설득력 개선 사항 통합;
            일관성, 유창성, 자연스러운 목소리 확보;
            전문적이고, 효과적이며, 읽기 쉬운 버전 완성.""",
)

critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""당신은 이메일 품질 평가자입니다.
            모든 응답은 반드시 한국어로 작성하세요.
            종합된 이메일을 최종 검토하고 전문적 기준을 충족하는지 판단하는 것이 당신의 역할입니다. 다음을 검토하세요:
            명확성과 흐름, 적절한 전문적 어조, 효과적인 행동 유도, 전반적 일관성.
            건설적이되 단호하게 평가하세요. 이메일에 중대한 결함(불명확한 메시지, 비전문적 어조,
            핵심 요소 누락)이 있으면, 구체적인 개선 제안을 하나만 제시하세요. 이메일이 전문적 기준을 충족하고 효과적으로 전달된다면, '이메일이 전문적 기준을 충족합니다.'라고 응답한 뒤 새 줄에 `TERMINATE`를 작성하세요. 실무에 쓸 수 있을 만큼 완벽한 이메일만 승인하고, 타협하지 마세요.""",
)

In [ ]:
text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)
termination_condition = text_termination | max_messages_termination

In [ ]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent,
    ],
    termination_condition=termination_condition,
)

await Console(
    team.run_stream(
        task="안녕하세요 이번주 목요일 티타임 시간에 AI 관련 교육을 진행하도록 하겠습니다."
    )
)